In [1]:
%matplotlib qt
%reload_ext autoreload
%autoreload 2
import pl_spec_python as pl_spec
import plotter
import sgd
import filter as fil
sgd.sgd_init()

Scanning for instruments...
Getting ready...
Ready for use!


In [3]:
import plotter



## Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `foldername` | str | Base name of the folder where data will be saved. |
| `scan_type` | str | Subfolder category. Options: `"coarse"` or `"fine"`. |
| `xdim`, `ydim` | float | Scan area width/height in um. Set to `0` for single-point. |
| `dx`, `dy` | float | Step size in um. |
| `center` | tuple | `(x, y)` centre of scan in um. |
| `grating` | int | `150` (150 g/mm) or `600` (600 g/mm). |
| `exposure_time` | float | Integration time per point in seconds. |
| `center_wavelength` | int | Spectrometer centre wavelength in nm. |
| `classification_threshold` | float | Min fraction of laser peak for emitter classification. |
| `current_user` | str | User for Telegram focus alerts. Options: `"shuhul"`, `"kristina"`, `"holland"`. |

In [83]:
foldername = '20260616-PLSPC-EL-BulkAnneal5262026-Ch1-T7A2-50uW-1s-1'
scan_type  = 'coarse'

pl_spec.pl_spec_lf(
    xdim=30, ydim=30, dx=0.5, dy=0.5, center=(0, 0),
    grating=150,
    exposure_time=1,
    center_wavelength=700,
    classification_threshold=1.05,
    foldername=foldername,
    scan_type=scan_type,
    current_user='holland'
)

plotter.plot_heatmap_manual(foldername, scan_type)

Connecting to LightField...
LightField bridge already running.
Getting wavelengths and setting up...
Starting scan...
Sgd on!


Acquiring x=15.00, y=15.00: 100%|██████████| 3721/3721 [2:39:23<00:00,  2.57s/it]   


Sgd off!
Scan complete, classifying data...
Found 119 distinct emitters.
Classification complete!


In [98]:
# Goto
sgd.goto(-1, 14.5)

Sgd on!
Moving to (-1 um, 14.5 um)
Done moving!


In [82]:
# End goto
sgd.home()

Moving to (0 um, 0 um)
Done moving!
Sgd off!


In [52]:

foldername = '20260615-PLSPC-EL-BulkAnneal5262026-Ch1-T8A1-50uW-1s-1'
scan_type  = 'coarse'
plotter.plot_heatmap_manual(foldername, scan_type)

In [4]:
FOLDERNAME   = '20260608-PLSPC-HT-Ch4-f2-500uW-1s-fullauto'
DATA_FOLDER  = 'data'
plotter.plot_heatmap_manual(FOLDERNAME, 'coarse', data_folder=DATA_FOLDER)

In [16]:
import filter as fil
print(fil.flipper)

None


In [17]:

  fil.filter_init()
  fil.filter_on()
  fil.flip_down()
  print(fil.flipper)
  if fil.flipper is not None:
      fil.filter_off()


Building device list for filter...
Connecting to Filter Flipper (Serial: 37010764)...


DeviceNotReadyException: Device is not connected
   at Thorlabs.MotionControl.DeviceManagerCLI.ThorlabsGenericCoreDeviceCLI.VerifyDeviceConnected(Int32 functionDepth)
   at Thorlabs.MotionControl.GenericMotorCLI.GenericMotorCLI.Connect(String serialNo)

Standalone G2



In [ ]:
G2_TARGET_RECORDS = 50_000_000   # stop after this many TTTR records
G2_TIME_NS        = 100.0        # correlation half-window (ns)
G2_TIMEBIN_NS     = 0.25         # bin width (ns)
OUT_FOLDER        = 'g2_data'    # folder for .npz and .png outputs
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
sys.path.insert(0, os.path.abspath('..'))   # adjust if needed to reach your modules

import picoharp
import g2 as g2mod


In [ ]:
picoharp.ph_init()

In [ ]:
for _ in range(3):
    r0, r1 = picoharp.get_count_rates()
    print(f'Ch0: {r0:.2e} cps   Ch1: {r1:.2e} cps')
    import time; time.sleep(1.0)

input('Press Enter to start acquisition...')

In [ ]:
npz_path = picoharp.ph_acquire(
    G2_TARGET_RECORDS,
    out_folder=OUT_FOLDER,
)
print(f'Raw data saved to: {npz_path}')

In [ ]:
if npz_path:
    result = g2mod.run(
        npz_path,
        out_folder=OUT_FOLDER,
        g2time_ns=G2_TIME_NS,
        timebin_ns=G2_TIMEBIN_NS,
    )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')          # remove this line if your kernel supports inline plots
from IPython.display import display, Image

stem   = os.path.splitext(os.path.basename(npz_path))[0]
png    = os.path.join(OUT_FOLDER, stem + '_plot.png')
display(Image(filename=png))

    # ── 6. Print key results ──────────────────────────────────────────────────
if result['popt'] is not None:
    a, b, T1, T2, g0 = result['popt']
    print(f"\ng²(0)      = {result['g2_0_norm']:.4f}")
    print(f"T1         = {T1:.2f} ns")
    print(f"Baseline   = {g0:.4f}")
    print(f"Wing level = {result['wing_level']:.1f} counts/bin")
    print(f"{'✓ SINGLE EMITTER' if result['g2_0_norm'] < 0.5 else '✗ Not a single emitter'}")
else:
    print("Fit did not converge.")


picoharp.ph_close()